In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("eda.ipynb"))
PNG_DIR = os.path.join(NOTEBOOK_DIR, "png")
os.makedirs(PNG_DIR, exist_ok=True)

def savefig(name, fig=None):
    f = fig or plt.gcf()
    f.savefig(os.path.join(PNG_DIR, name), dpi=300, bbox_inches="tight")
    plt.close(f)

In [2]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("eda.ipynb"))
BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

data = os.path.join(BASE_DIR, "data", "concat_for_eda.csv")

if not os.path.exists(data):
    print(f"[!] {data} not found.")
else:
    print(f"Loading: {data}")

Loading: C:\Users\AGG8HC\Desktop\Project\microcytic-reduced\data\concat_for_eda.csv


In [3]:
df= pd.read_csv(data)
print(df)
print(df.shape)

     hemoglobin   MCV   Fe  Ferritin  Transferrin  tsat    CRP anemia_class  \
0            86  76.6  3.1    1062.0        170.0   2.9  174.8      ACD/IDA   
1           114  78.5  2.2     457.0        145.0   8.5   15.3          ACD   
2            84  75.5  2.6     445.3         93.0  46.3  120.1          ACD   
3           115  79.6  1.7    1151.0        125.0   7.6    9.9          ACD   
4            90  57.3  2.9    2009.0        116.0  14.0   62.9          ACD   
..          ...   ...  ...       ...          ...   ...    ...          ...   
247          75  62.8  2.4       9.2          NaN   NaN    NaN          IDA   
248          87  65.7  NaN       NaN          NaN   NaN    NaN          IDA   
249          89  73.3  5.7     111.1          NaN   NaN    0.7          IDA   
250          84  70.1  NaN       NaN          NaN   NaN    NaN          IDA   
251          82  68.1  NaN       NaN          NaN   NaN    NaN          IDA   

     history_ida  history_acd  ...  HbA2  HbF  HbH 

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   hemoglobin     252 non-null    int64  
 1   MCV            252 non-null    float64
 2   Fe             239 non-null    float64
 3   Ferritin       236 non-null    float64
 4   Transferrin    166 non-null    float64
 5   tsat           176 non-null    float64
 6   CRP            181 non-null    float64
 7   anemia_class   252 non-null    str    
 8   history_ida    252 non-null    bool   
 9   history_acd    252 non-null    bool   
 10  RBC            113 non-null    float64
 11  MCHC           116 non-null    float64
 12  rdw_cv         82 non-null     float64
 13  TIBC           1 non-null      float64
 14  ret_he         7 non-null      float64
 15  HbA1           85 non-null     float64
 16  HbA2           75 non-null     float64
 17  HbF            1 non-null      float64
 18  HbH            0 non-

### Descriptive Statistics by Diagnosis Group

In [5]:
df["anemia_class"].unique().tolist()

['ACD/IDA', 'ACD', 'IDA']

In [6]:
labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

for label in labels:
    subset = df[df["anemia_class"] == label][numeric_cols]
    n = len(subset)
    print(f"{label} (n={n})")
    print(subset.describe().round(2).to_string())

IDA (n=109)
          CRP     Fe  Ferritin   HbA1   HbA2  HbBart    HbE  HbF  HbH  HbS    MCHC     MCV     RBC  TIBC  Transferrin  hb_other  hemoglobin  rdw_cv  ret_he   tsat
count   40.00  96.00     94.00  78.00  69.00     0.0   3.00  1.0  0.0  0.0  109.00  109.00  106.00   1.0        27.00       0.0      109.00   80.00    5.00  36.00
mean    10.54   3.64     20.29  97.48   2.00     NaN  13.27  0.8  NaN  NaN  282.11   62.65    3.60  73.4       340.00       NaN       64.25   20.85   19.21   8.72
std     22.65   4.12     24.82   2.97   0.48     NaN   7.05  NaN  NaN  NaN   20.64    7.87    0.93   NaN        58.77       NaN       19.53    2.49   32.56  13.81
min      0.20   0.69      1.93  77.70   1.50     NaN   5.80  0.8  NaN  NaN  241.00   45.00    1.23  73.4       200.00       NaN       17.00   13.60    1.45   1.50
25%      0.40   2.00      5.03  97.90   1.80     NaN  10.00  0.8  NaN  NaN  266.00   55.90    2.98  73.4       325.00       NaN       52.00   19.10    1.89   3.00
50%      2

### Missing Value Rate by Diagnosis Group
Heatmap showing the percentage of missing values for each feature within each diagnosis group. Supports the argument for not imputing — each diagnosis group has a different missing data pattern.

In [7]:
labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

missing_by_diag = pd.DataFrame()
for label in labels:
    subset = df[df["anemia_class"] == label][numeric_cols]
    missing_pct = (subset.isnull().sum() / len(subset) * 100).round(1)
    missing_by_diag[label] = missing_pct

missing_by_diag_filtered = missing_by_diag[missing_by_diag.max(axis=1) > 0]

if missing_by_diag_filtered.empty:
    print("No missing values found — showing full missing-rate table (all zeros).")
    plt.figure(figsize=(10, 8))
    sns.heatmap(missing_by_diag, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5,
                vmin=0, vmax=1)
    plt.title("Missing Value Rate (%) by Diagnosis Group")
    plt.ylabel("Feature")
    plt.tight_layout()
    savefig("missing_by_diagnosis.png")
else:
    plt.figure(figsize=(10, 8))
    sns.heatmap(missing_by_diag_filtered, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5)
    plt.title("Missing Value Rate (%) by Diagnosis Group")
    plt.ylabel("Feature")
    plt.tight_layout()
    savefig("missing_by_diagnosis.png")

print(missing_by_diag)

               IDA    ACD  ACD/IDA
CRP           63.3    0.0     10.0
Fe            11.9    0.0      0.0
Ferritin      13.8    0.0      5.0
HbA1          28.4   99.2     70.0
HbA2          36.7   99.2     75.0
HbBart       100.0  100.0    100.0
HbE           97.2  100.0    100.0
HbF           99.1  100.0    100.0
HbH          100.0  100.0    100.0
HbS          100.0  100.0    100.0
MCHC           0.0   99.2     70.0
MCV            0.0    0.0      0.0
RBC            2.8   99.2     70.0
TIBC          99.1  100.0    100.0
Transferrin   75.2    0.0     20.0
hb_other     100.0  100.0    100.0
hemoglobin     0.0    0.0      0.0
rdw_cv        26.6   99.2     95.0
ret_he        95.4  100.0     90.0
tsat          67.0    0.0     15.0


### Shapiro-Wilk Normality Test
Test normality of each feature within each diagnosis group. If most groups reject normality (p<0.05), this justifies using non-parametric tests (Kruskal-Wallis) instead of ANOVA.

In [8]:
from scipy.stats import shapiro
import warnings

labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

shapiro_results = []

for col in numeric_cols:
    for label in labels:
        vals = df[df["anemia_class"] == label][col].dropna()
        if len(vals) >= 3:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                stat, p = shapiro(vals)
            shapiro_results.append({
                "Feature": col,
                "Group": label,
                "W-statistic": round(stat, 4),
                "p-value": round(p, 6),
                "Normal": "Yes" if p >= 0.05 else "No"
            })

shapiro_df = pd.DataFrame(shapiro_results)

non_normal_count = shapiro_df[shapiro_df["Normal"] == "No"].shape[0]
total_tests = shapiro_df.shape[0]
print(f"{non_normal_count}/{total_tests} tests reject normality (p<0.05)")
print(f"→ Justifies using Kruskal-Wallis (non-parametric) instead of ANOVA\n")

summary = shapiro_df.pivot_table(index="Feature", columns="Group", values="Normal", aggfunc="first")
print(summary)

15/32 tests reject normality (p<0.05)
→ Justifies using Kruskal-Wallis (non-parametric) instead of ANOVA

Group        ACD ACD/IDA  IDA
Feature                      
CRP           No      No   No
Fe           Yes      No   No
Ferritin      No      No   No
HbA1         NaN     Yes   No
HbA2         NaN     Yes   No
HbE          NaN     NaN  Yes
MCHC         NaN     Yes  Yes
MCV           No     Yes  Yes
RBC          NaN     Yes  Yes
Transferrin  Yes     Yes  Yes
hemoglobin   Yes     Yes  Yes
rdw_cv       NaN     NaN  Yes
ret_he       NaN     NaN   No
tsat          No      No   No


### Kruskal-Wallis Test
Compare the distribution of each feature across the diagnosis groups

In [9]:
from scipy.stats import kruskal
import warnings

labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

kw_results = []

for col in numeric_cols:
    groups = []
    for label in labels:
        vals = df[df["anemia_class"] == label][col].dropna()
        if len(vals) >= 3:
            groups.append(vals)

    if len(groups) >= 2:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            stat, p = kruskal(*groups)

        if not pd.isna(stat):
            kw_results.append({
                "Feature": col,
                "H-statistic": round(stat, 4),
                "p-value": round(p, 6),
                "Significant": "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            })

kw_df = pd.DataFrame(kw_results).sort_values("p-value")
print(kw_df.to_string(index=False))

sig_features = kw_df[kw_df["Significant"] != ""]["Feature"].tolist()
print(f"\nStatistically significant features (p<0.05): {sig_features}")

    Feature  H-statistic  p-value Significant
        CRP      67.6633 0.000000         ***
   Ferritin     162.7813 0.000000         ***
        MCV      78.0852 0.000000         ***
Transferrin      66.0682 0.000000         ***
 hemoglobin      83.3958 0.000000         ***
       tsat      35.6618 0.000000         ***
         Fe      20.4724 0.000036         ***
       HbA2       0.6807 0.409332            
       HbA1       0.5794 0.446530            
       MCHC       0.4966 0.481005            
        RBC       0.4010 0.526596            

Statistically significant features (p<0.05): ['CRP', 'Ferritin', 'MCV', 'Transferrin', 'hemoglobin', 'tsat', 'Fe']


In [10]:
import scikit_posthocs as sp
import warnings

labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

sig_features = kw_df[kw_df["Significant"] != ""]["Feature"].tolist()

for col in sig_features:
    melted = []
    for label in labels:
        vals = df[df["anemia_class"] == label][col].dropna()
        for v in vals:
            melted.append({"Group": label, "Value": v})

    melted_df = pd.DataFrame(melted)

    if melted_df.empty or melted_df["Group"].nunique() < 2:
        continue

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dunn = sp.posthoc_dunn(melted_df, val_col="Value", group_col="Group", p_adjust="bonferroni")

    sig_pairs = []
    for i in range(len(dunn.index)):
        for j in range(i + 1, len(dunn.columns)):
            g1, g2 = dunn.index[i], dunn.columns[j]
            p = dunn.iloc[i, j]
            if p < 0.05:
                star = "***" if p < 0.001 else "**" if p < 0.01 else "*"
                sig_pairs.append(f"  {g1} vs {g2}: p={p:.6f} {star}")

    if sig_pairs:
        print(f"\n{col}")
        for pair in sig_pairs:
            print(pair)


CRP
  ACD vs IDA: p=0.000000 ***
  ACD/IDA vs IDA: p=0.000000 ***

Ferritin
  ACD vs IDA: p=0.000000 ***
  ACD/IDA vs IDA: p=0.000004 ***

MCV
  ACD vs IDA: p=0.000000 ***
  ACD/IDA vs IDA: p=0.006642 **

Transferrin
  ACD vs IDA: p=0.000000 ***
  ACD/IDA vs IDA: p=0.000046 ***

hemoglobin
  ACD vs IDA: p=0.000000 ***
  ACD/IDA vs IDA: p=0.020848 *

tsat
  ACD vs IDA: p=0.000000 ***

Fe
  ACD vs ACD/IDA: p=0.008427 **
  ACD vs IDA: p=0.000166 ***


### Univariable Distribution

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
df_copy= df.copy()

X_y_don= df_copy.drop(columns= ["ACD", "IDA"])


In [13]:
null_X = X_y_don.drop(columns=["anemia_class"])

null_counts = null_X.notnull().sum()

plt.figure(figsize=(10, 5))
null_counts.plot(kind='bar')

plt.xticks(rotation=80)

plt.title('Number of Non-null values per Column')
plt.xlabel('Columns')
plt.ylabel('Count of Non-null Values')

plt.tight_layout()
savefig("nonnull_counts.png")

In [14]:
null_X = X_y_don.drop(columns=["anemia_class"])

null_counts = null_X.notnull().sum()

gender_counts = X_y_don["gender_female"].value_counts()
gender_counts.index = ["Female" if v else "Male" for v in gender_counts.index]

null_counts = null_counts.drop("gender_female")
null_counts = pd.concat([null_counts, gender_counts])

plt.figure(figsize=(10, 5))
null_counts.plot(kind="bar")

plt.xticks(rotation=80)

plt.title("Number of Non-null values per Column (with Gender breakdown)")
plt.xlabel("Columns")
plt.ylabel("Count")

plt.tight_layout()
savefig("nonnull_with_gender.png")

In [15]:
high_missing = X_y_don.drop(columns=["anemia_class"]).columns[X_y_don.drop(columns=["anemia_class"]).isnull().mean() > 0.5].tolist()
drop_cols = [c for c in high_missing + ["gender_female"] if c in X_y_don.columns]
X_y_don_notnull = X_y_don.drop(columns=drop_cols)

for col_name in X_y_don_notnull.columns:
    if col_name != "anemia_class":
        fig, ax = plt.subplots()
        sns.boxplot(
            data=X_y_don_notnull,
            x="anemia_class",
            y=col_name,
            hue="anemia_class",
            legend=False,
            ax=ax
        )
        ax.set_title(f"Boxplot: {col_name} by Diagnosis")
        savefig(f"boxplot_{col_name}.png", fig)

In [16]:
fig, ax = plt.subplots()
gender_labels = X_y_don["gender_female"].map({True: "Female", False: "Male"})
sns.countplot(x=gender_labels, ax=ax)
ax.set_xlabel("Gender")
ax.set_ylabel("Count")
ax.set_title("Gender Distribution")
savefig("gender_distribution.png", fig)

In [17]:
plot_df = X_y_don.copy()
plot_df["gender"] = plot_df["gender_female"].map({True: "Female", False: "Male"})

fig, ax = plt.subplots()
sns.countplot(data=plot_df, x="gender", hue="anemia_class", ax=ax)
ax.legend(
    title="Distribution of gender by diagnosis",
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)
ax.set_title("Gender by Diagnosis Group")
savefig("gender_by_diagnosis.png", fig)

### Correlation Matrix — CBC Parameters Only

In [18]:
cbc_cols = ["RBC", "hemoglobin", "MCV", "MCHC", "rdw_cv"]
cbc_available = [c for c in cbc_cols if c in df.columns]

corr_cbc = df[cbc_available].corr()

n_matrix_cbc = pd.DataFrame(index=cbc_available, columns=cbc_available, dtype=int)
for c1 in cbc_available:
    for c2 in cbc_available:
        n_matrix_cbc.loc[c1, c2] = df[[c1, c2]].dropna().shape[0]

annot_cbc = corr_cbc.copy().astype(str)
for c1 in cbc_available:
    for c2 in cbc_available:
        r = corr_cbc.loc[c1, c2]
        n = n_matrix_cbc.loc[c1, c2]
        annot_cbc.loc[c1, c2] = f"{r:.2f}\n(n={n})"

mask = np.triu(np.ones_like(corr_cbc, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_cbc, mask=mask, annot=annot_cbc, fmt="",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax,
    cbar_kws={"shrink": 0.8}, annot_kws={"size": 9}
)
ax.set_title("Correlation Matrix — CBC Parameters (with pairwise n)")
savefig("corr_cbc_only.png", fig)

print("Pairwise sample sizes:")
print(n_matrix_cbc.to_string())

Pairwise sample sizes:
              RBC  hemoglobin    MCV   MCHC  rdw_cv
RBC         113.0       113.0  113.0  113.0    81.0
hemoglobin  113.0       252.0  252.0  116.0    82.0
MCV         113.0       252.0  252.0  116.0    82.0
MCHC        113.0       116.0  116.0  116.0    82.0
rdw_cv       81.0        82.0   82.0   82.0    82.0


### Correlation Matrix — CBC + Iron/Inflammation Parameters

In [19]:
iron_cols = ["Ferritin", "CRP", "Fe", "TIBC", "tsat"]
all_cols = cbc_cols + iron_cols
all_available = [c for c in all_cols if c in df.columns]

corr_full = df[all_available].corr()

n_matrix_full = pd.DataFrame(index=all_available, columns=all_available, dtype=int)
for c1 in all_available:
    for c2 in all_available:
        n_matrix_full.loc[c1, c2] = df[[c1, c2]].dropna().shape[0]

annot_full = corr_full.copy().astype(str)
for c1 in all_available:
    for c2 in all_available:
        r = corr_full.loc[c1, c2]
        n = n_matrix_full.loc[c1, c2]
        if pd.isna(r):
            annot_full.loc[c1, c2] = f"NaN\n(n={n})"
        else:
            annot_full.loc[c1, c2] = f"{r:.2f}\n(n={n})"

mask_full = np.triu(np.ones_like(corr_full, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    corr_full, mask=mask_full, annot=annot_full, fmt="",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax,
    cbar_kws={"shrink": 0.8}, annot_kws={"size": 8}
)
ax.set_title("Correlation Matrix — CBC + Iron/Inflammation (with pairwise n)")
savefig("corr_cbc_iron.png", fig)

print("Pairwise sample sizes:")
print(n_matrix_full.to_string())

Pairwise sample sizes:
              RBC  hemoglobin    MCV   MCHC  rdw_cv  Ferritin    CRP     Fe  TIBC   tsat
RBC         113.0       113.0  113.0  113.0    81.0      99.0   45.0  103.0   1.0   38.0
hemoglobin  113.0       252.0  252.0  116.0    82.0     236.0  181.0  239.0   1.0  176.0
MCV         113.0       252.0  252.0  116.0    82.0     236.0  181.0  239.0   1.0  176.0
MCHC        113.0       116.0  116.0  116.0    82.0     100.0   45.0  103.0   1.0   40.0
rdw_cv       81.0        82.0   82.0   82.0    82.0      77.0   30.0   77.0   0.0   28.0
Ferritin     99.0       236.0  236.0  100.0    77.0     236.0  179.0  235.0   1.0  172.0
CRP          45.0       181.0  181.0   45.0    30.0     179.0  181.0  179.0   1.0  150.0
Fe          103.0       239.0  239.0  103.0    77.0     235.0  179.0  239.0   1.0  173.0
TIBC          1.0         1.0    1.0    1.0     0.0       1.0    1.0    1.0   1.0    0.0
tsat         38.0       176.0  176.0   40.0    28.0     172.0  150.0  173.0   0.0  176.

In [20]:
fig, ax = plt.subplots()
sns.histplot(df["anemia_class"], ax=ax)
ax.set_title("Diagnosis Class Distribution")
plt.xticks(rotation=0)
savefig("diagnosis_distribution.png", fig)

In [21]:
y = df[["ACD", "IDA"]]

counts = y.value_counts()

print(counts)

ACD  IDA
1    0      123
0    1      109
1    1       20
Name: count, dtype: int64


In [22]:
labels_binary = ["ACD", "IDA"]

co_matrix = pd.DataFrame(0, index=labels_binary, columns=labels_binary)

for _, row in y.iterrows():
    active_labels = row[row == 1].index.tolist()
    for l1 in active_labels:
        for l2 in active_labels:
            co_matrix.loc[l1, l2] += 1

print("Co-occurrence matrix:")
print(co_matrix)

Co-occurrence matrix:
     ACD  IDA
ACD  143   20
IDA   20  129


### Model Comparison — Friedman Test
Train all models (XGBoost, LightGBM, CatBoost), collect CV fold scores, and run Friedman test + Nemenyi post-hoc to compare.

In [23]:
import sys, os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

train_path = os.path.join(BASE_DIR, "data", "train_set_reduced_features.csv")
test_path = os.path.join(BASE_DIR, "data", "test_set_reduced_features.csv")

if not os.path.exists(train_path) or not os.path.exists(test_path):
    print(f"[skip] Training/test data not found. Run train/test split first.")
else:
    from src.dataclass.modelfactory import ModelFactory, models_config
    from src.functions.function import friedman_compare, print_result

    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)

    X_train = df_train.drop(columns=["ACD", "IDA"])
    y_train = df_train[["ACD", "IDA"]]
    X_test = df_test.drop(columns=["ACD", "IDA"])
    y_test = df_test[["ACD", "IDA"]]

    all_cv_scores = {}

    for name in models_config:
        print(f"\n{'='*40}")
        print(f"Training: {name}")
        print(f"{'='*40}")
        factory = ModelFactory(name)
        result = factory.train_and_evaluate(X_train, y_train, X_test, y_test)
        print_result(result)
        all_cv_scores[name] = result["cv_fold_scores"]

    print(f"\n{'='*60}")
    print("Friedman Test — Comparing all models")
    print(f"{'='*60}")
    friedman_compare(all_cv_scores)

[skip] Training/test data not found. Run train/test split first.
